In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import RandomOverSampler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🟢 Using device: {device}")

🟢 Using device: cuda


In [4]:
df = pd.read_csv("./Data/creditcard.csv",)

In [5]:
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [6]:
if "Time" in df.columns:
    df = df.drop("Time", axis=1)

In [7]:
df.head()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [9]:
X = df.drop("Class", axis=1)
y = df["Class"]

In [10]:
ros = RandomOverSampler(random_state=42)
X_res, y_res = ros.fit_resample(X, y)

In [11]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_res)

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_res, test_size=0.2, random_state=42
)

In [13]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1).to(device)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1).to(device)

In [14]:
class CreditNN(nn.Module):
    def __init__(self, input_dim):
        super(CreditNN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

In [15]:
model = CreditNN(X_train.shape[1]).to(device)

In [16]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [17]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [18]:
epochs = 50
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss/len(train_loader):.4f}")

Epoch [1/50], Loss: 0.0788
Epoch [2/50], Loss: 0.0323
Epoch [3/50], Loss: 0.0236
Epoch [4/50], Loss: 0.0212
Epoch [5/50], Loss: 0.0189
Epoch [6/50], Loss: 0.0182
Epoch [7/50], Loss: 0.0165
Epoch [8/50], Loss: 0.0164
Epoch [9/50], Loss: 0.0162
Epoch [10/50], Loss: 0.0166
Epoch [11/50], Loss: 0.0161
Epoch [12/50], Loss: 0.0155
Epoch [13/50], Loss: 0.0153
Epoch [14/50], Loss: 0.0158
Epoch [15/50], Loss: 0.0165
Epoch [16/50], Loss: 0.0152
Epoch [17/50], Loss: 0.0155
Epoch [18/50], Loss: 0.0149
Epoch [19/50], Loss: 0.0144
Epoch [20/50], Loss: 0.0143
Epoch [21/50], Loss: 0.0144
Epoch [22/50], Loss: 0.0157
Epoch [23/50], Loss: 0.0142
Epoch [24/50], Loss: 0.0149
Epoch [25/50], Loss: 0.0148
Epoch [26/50], Loss: 0.0137
Epoch [27/50], Loss: 0.0146
Epoch [28/50], Loss: 0.0140
Epoch [29/50], Loss: 0.0143
Epoch [30/50], Loss: 0.0142
Epoch [31/50], Loss: 0.0142
Epoch [32/50], Loss: 0.0142
Epoch [33/50], Loss: 0.0140
Epoch [34/50], Loss: 0.0142
Epoch [35/50], Loss: 0.0139
Epoch [36/50], Loss: 0.0141
E

In [19]:
from sklearn.metrics import classification_report

# Set model to evaluation mode
model.eval()

# Disable gradient calculation for evaluation
with torch.no_grad():
    # Get predictions on the test set
    y_pred_probs = model(X_test_tensor)
    y_pred = (y_pred_probs >= 0.5).float()  # Convert probabilities to 0 or 1

# Move tensors to CPU before converting to numpy
y_true_np = y_test_tensor.cpu().numpy()
y_pred_np = y_pred.cpu().numpy()

# Print classification report
print(classification_report(y_true_np, y_pred_np, target_names=["0", "1"]))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56750
           1       1.00      1.00      1.00     56976

    accuracy                           1.00    113726
   macro avg       1.00      1.00      1.00    113726
weighted avg       1.00      1.00      1.00    113726



In [22]:
torch.save({
    "model_state_dict": model.state_dict(),
    "scaler": scaler
}, "./Models/Credit_fraud.pth")

print("\n✅ Neural network trained on GPU and saved as 'Credit_fraud.pth'.")



✅ Neural network trained on GPU and saved as 'Credit_fraud.pth'.
